# Hansen Ch.15 Multivariate Time Series

理论见 md（**15.1–15.20**）。本 notebook：平稳性核对 + 主要实证。

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv, eigvals
from scipy.linalg import cholesky
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def max_abs_eig(A):
    return float(np.max(np.abs(eigvals(A))))

# 15.1-15.2
print("15.1a", max_abs_eig([[0.7,0.2],[0.2,0.7]]))
print("15.1b", max_abs_eig([[0.8,0.4],[0.4,0.8]]))
print("15.1c", max_abs_eig([[0.8,0.4],[-0.4,0.8]]))
A1=np.array([[0.3,0.2],[0.2,0.3]]); A2=np.array([[0.4,-0.1],[-0.1,0.4]])
C=np.block([[A1,A2],[np.eye(2),np.zeros((2,2))]])
print("15.2 companion", max_abs_eig(C))

# 15.12
rho=0.8
Sig=np.array([[1.,rho],[rho,1.]])
L=cholesky(Sig, lower=True)
print("15.12 L\n", L)
Theta=np.array([[1.,0.],[1.,1.]])
print("OIRF\n", Theta@L)


In [ ]:

def make_lags(Y, p):
    T, m = Y.shape
    y = Y[p:]
    parts = [np.ones((T-p,1))]
    for j in range(1,p+1):
        parts.append(Y[p-j:T-j])
    return y, np.hstack(parts)

def estimate_var(Y, p):
    y, X = make_lags(Y, p)
    B = inv(X.T@X)@(X.T@y)
    E = y - X@B
    return B, E, E.T@E/len(y), len(y)

def companion_irf(B, p, m, hmax, shock):
    As = [B[1+(j-1)*m:1+j*m,:].T for j in range(1,p+1)]
    k=m*p
    C=np.zeros((k,k))
    C[0:m,0:m]=As[0]
    for j in range(1,p):
        C[0:m, j*m:(j+1)*m]=As[j]
    if p>1: C[m:,0:m*(p-1)]=np.eye(m*(p-1))
    irfs=[]; state=np.zeros(k); state[:m]=shock
    for h in range(hmax+1):
        irfs.append(state[:m].copy()); state=C@state
    return np.array(irfs)

def aic_var(Y,p):
    y,X=make_lags(Y,p)
    B=inv(X.T@X)@(X.T@y)
    E=y-X@B
    n,m=y.shape
    Sig=E.T@E/n
    return np.log(np.linalg.det(Sig))+2*(m*m*p+m)/n


## 15.14–15.16, 15.19–15.20 实证摘要

In [ ]:

qd=pd.read_excel(ROOT/"FRED-QD/FRED-QD.xlsx")
md=pd.read_excel(ROOT/"FRED-MD/FRED-MD.xlsx")
# 15.14
g=100*np.log(pd.to_numeric(qd["gdpc1"],errors="coerce")).diff()
pi=100*np.log(pd.to_numeric(qd["gdpctpi"],errors="coerce")).diff()
ff=pd.to_numeric(qd["fedfunds"],errors="coerce")
Y=pd.DataFrame({"g":g,"pi":pi,"ff":ff}).dropna().values
B,E,Sig,n=estimate_var(Y,6)
L=cholesky(Sig,lower=True)
irf=companion_irf(B,6,3,12,L[:,0])
print("15.14 n",n,"cum GDP supply",np.cumsum(irf[:,0])[[0,4,8,12]])

# Kilian
kil=pd.read_excel(ROOT/"Kilian2009/Kilian2009.xlsx")
Yk=kil[["oil","output","price"]].astype(float).values
Yk[:,0]*=-1
B,E,Sig,n=estimate_var(Yk,4)
L=cholesky(Sig,lower=True)
print("15.15 Kilian n",n)
for s in range(3):
    irf=companion_irf(B,4,3,12,L[:,s])
    print(" output IRF shock",s, np.round(irf[[0,6,12],1],3))

# housing
for c in ["permit","houst","realln"]:
    md[c]=pd.to_numeric(md[c],errors="coerce")
df=pd.DataFrame({"permit":md.permit,"houst":md.houst,"gloan":100*np.log(md.realln).diff()}).dropna()
aics=[(p,aic_var(df.values,p)) for p in range(1,9)]
print("15.16 AIC best", min(aics,key=lambda x:x[1]))

# 15.19 Granger
gdp=100*np.log(pd.to_numeric(qd["gdpc1"],errors="coerce")).diff()
m1n=pd.to_numeric(qd["m1realx"],errors="coerce")*pd.to_numeric(qd["cpiaucsl"],errors="coerce")
mg=100*np.log(m1n).diff()
d=pd.DataFrame({"g":gdp,"m":mg}).dropna()
g,m=d.g.values,d.m.values
p=4
Y=g[p:]
X=np.column_stack([g[p-j:len(g)-j] for j in range(1,5)]+[m[p-j:len(m)-j] for j in range(1,5)]+[np.ones(len(Y))])
b=inv(X.T@X)@(X.T@Y)
e=Y-X@b
V=inv(X.T@X)@((X*e[:,None]).T@(X*e[:,None]))@inv(X.T@X)
R=np.zeros((4,9))
for i in range(4): R[i,4+i]=1
W=float((R@b)@inv(R@V@R.T)@(R@b))
print("15.19 Granger money W",W,"p",1-stats.chi2.cdf(W,4))
print("sum money", b[4:8].sum())
